In [9]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle

# -----------------------------
# User parameters
# -----------------------------
BUNDLE_DIAMETER_UM = 75.0
BUNDLE_RADIUS_UM = BUNDLE_DIAMETER_UM / 2

CORE_SPACING_UM = 3.3        # center-to-center distance
CORE_DIAMETER_UM = 3.0       # core diameter
CORE_RADIUS_UM = CORE_DIAMETER_UM / 2

HEX_RADIUS = 1               # 1 = 7 blue circles: center + 6 neighbors

# Blue region position
# Negative y moves blue cluster downward.
BLUE_CENTER_X_UM = 0.0
BLUE_CENTER_Y_UM = -BUNDLE_RADIUS_UM * 0.38

OUTPUT_SVG = "multicore_fiber_cross_section.svg"

# -----------------------------
# Colors matching the figure
# -----------------------------
FIGURE_BACKGROUND_COLOR = "#ffffff"   # outside the circular bundle
BUNDLE_BACKGROUND_COLOR = "#4a4a4a"   # dark gray inside circle
CORE_COLOR = "#ffffff"                # white cores
BLUE_COLOR = "#2137ff"                # blue highlighted cores
CONTOUR_COLOR = "#000000"             # black circular contour
ANNOTATION_COLOR = "#0c0c0c"          # white arrow/text inside dark bundle

# -----------------------------
# Generate hex-packed cores
# -----------------------------
dx = CORE_SPACING_UM
dy = CORE_SPACING_UM * np.sqrt(3) / 2

# Keep core centers only where the full core fits inside the bundle
max_center_radius = BUNDLE_RADIUS_UM - CORE_RADIUS_UM

positions = []

j_max = int(np.ceil(max_center_radius / dy)) + 3
i_max = int(np.ceil(max_center_radius / dx)) + 3

for j in range(-j_max, j_max + 1):
    row_offset = 0.5 * dx if j % 2 else 0.0

    for i in range(-i_max, i_max + 1):
        x = i * dx + row_offset
        y = j * dy

        r = np.sqrt(x**2 + y**2)

        if r <= max_center_radius:
            positions.append((x, y, i, j, r))

positions = np.array(positions, dtype=float)

# -----------------------------
# Select offset blue 7-core hexagon
# -----------------------------
dist_to_requested_blue_center = np.sqrt(
    (positions[:, 0] - BLUE_CENTER_X_UM) ** 2
    + (positions[:, 1] - BLUE_CENTER_Y_UM) ** 2
)

blue_center_core = positions[np.argmin(dist_to_requested_blue_center)]
blue_center_x = blue_center_core[0]
blue_center_y = blue_center_core[1]

dist_to_blue_center = np.sqrt(
    (positions[:, 0] - blue_center_x) ** 2
    + (positions[:, 1] - blue_center_y) ** 2
)

highlight_mask = dist_to_blue_center <= CORE_SPACING_UM * HEX_RADIUS * 1.05

# -----------------------------
# Plot
# -----------------------------
fig, ax = plt.subplots(figsize=(0.8, 0.8))
fig.patch.set_facecolor(FIGURE_BACKGROUND_COLOR)
ax.set_facecolor(FIGURE_BACKGROUND_COLOR)

# Dark circular fiber/bundle background
bundle_fill = Circle(
    (0, 0),
    BUNDLE_RADIUS_UM,
    facecolor=BUNDLE_BACKGROUND_COLOR,
    edgecolor="none",
    linewidth=0,
)
ax.add_patch(bundle_fill)

# White cores
for x, y, i, j, r in positions[~highlight_mask]:
    ax.add_patch(
        Circle(
            (x, y),
            CORE_RADIUS_UM,
            facecolor=CORE_COLOR,
            edgecolor="none",
            linewidth=0,
        )
    )

# Blue offset hexagon
for x, y, i, j, r in positions[highlight_mask]:
    ax.add_patch(
        Circle(
            (x, y),
            CORE_RADIUS_UM,
            facecolor=BLUE_COLOR,
            edgecolor="none",
            linewidth=0,
        )
    )

# Black circular contour on top
bundle_outline = Circle(
    (0, 0),
    BUNDLE_RADIUS_UM,
    facecolor="none",
    edgecolor=CONTOUR_COLOR,
    linewidth=0.5,
)
ax.add_patch(bundle_outline)

# -----------------------------
# Spacing annotation
# -----------------------------
# Pick two adjacent cores near the upper central part of the bundle.
best_pair = None
best_score = np.inf

for a in range(len(positions)):
    for b in range(a + 1, len(positions)):
        x1, y1 = positions[a, 0], positions[a, 1]
        x2, y2 = positions[b, 0], positions[b, 1]

        d = np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

        if abs(d - CORE_SPACING_UM) < 0.03:
            mid_x = (x1 + x2) / 2
            mid_y = (y1 + y2) / 2

            target_y = BUNDLE_RADIUS_UM * 0.62

            # Prefer an almost-horizontal pair near top-center
            horizontal_penalty = abs(y2 - y1) * 10
            score = abs(mid_x) + abs(mid_y - target_y) + horizontal_penalty

            if score < best_score:
                best_score = score
                best_pair = (x1, y1, x2, y2)

if best_pair is not None:
    # Annotation and spacing label removed per user request
    pass

# -----------------------------
# Formatting
# -----------------------------
pad = CORE_RADIUS_UM * 2.0

ax.set_xlim(-BUNDLE_RADIUS_UM - pad, BUNDLE_RADIUS_UM + pad)
ax.set_ylim(-BUNDLE_RADIUS_UM - pad, BUNDLE_RADIUS_UM + pad)

ax.set_aspect("equal")
ax.axis("off")

plt.savefig(
    OUTPUT_SVG,
    format="svg",
    bbox_inches="tight",
    pad_inches=0.02,
    facecolor=fig.get_facecolor(),
)

plt.close(fig)

print(f"Saved SVG to: {OUTPUT_SVG}")
print(f"Total cores drawn: {len(positions)}")
print(f"Highlighted blue cores: {highlight_mask.sum()}")
print(f"Blue hexagon center: ({blue_center_x:.2f}, {blue_center_y:.2f}) µm")
print(f"Bundle diameter: {BUNDLE_DIAMETER_UM:g} µm")

Saved SVG to: multicore_fiber_cross_section.svg
Total cores drawn: 433
Highlighted blue cores: 7
Blue hexagon center: (-1.65, -14.29) µm
Bundle diameter: 75 µm
